In [1]:
# ==================================================================
# ЗАДАНИЯ ДЛЯ ПРАКТИЧЕСКОЙ РАБОТЫ К ГЛАВЕ 2
# Вариант 5:   ln(x + 5) = cos x        (по условию x < 5)
# ==================================================================
# Задание 1 — отделить корни графически, программно и в пакете
# Задание 2 — половинное деление, точность 1e-3 (вручную и программой)
# Задание 3 — простая итерация, точность 1e-6
# Задание 4 — комбинированный метод хорд и касательных, точность 1e-6
# Задание 5 — инструментальный пакет, точность 1e-6, сопоставление

import math

# ------- исследуемая функция и её производные -------
f = lambda x: math.log(x + 5) - math.cos(x)          # f(x) = ln(x+5) − cos x
d1 = lambda x: 1 / (x + 5) + math.sin(x)             # f′(x)
d2 = lambda x: -1 / (x + 5) ** 2 + math.cos(x)       # f″(x)

RESULTS = {}          # сюда каждое задание кладёт свой ответ


def plot(g, x0, x1, w=68, h=19, title=""):
    xs = [x0 + (x1 - x0) * i / (w - 1) for i in range(w)]
    ys = []
    for x in xs:
        try:
            ys.append(g(x))
        except (ValueError, ZeroDivisionError):
            ys.append(None)
    good = [v for v in ys if v is not None]
    lo, hi = min(good), max(good)
    pad = (hi - lo) * 0.05 or 1.0
    lo, hi = lo - pad, hi + pad
    row = lambda v: int(round((hi - v) / (hi - lo) * (h - 1)))
    grid = [[" "] * w for _ in range(h)]
    if lo <= 0 <= hi:
        for c in range(w):
            grid[row(0.0)][c] = "-"
    for c, v in enumerate(ys):
        if v is None:
            continue
        r = max(0, min(h - 1, row(v)))
        grid[r][c] = "*"
    if title:
        print(title)
    for r in range(h):
        print(f"{hi - (hi - lo) * r / (h - 1):+9.3f} |" + "".join(grid[r]))
    print(" " * 10 + "+" + "-" * w)
    print(" " * 11 + f"{x0:<{w // 2}.4g}{x1:>{w // 2}.4g}")


def tabulate(g, x0, x1, step, show=True):
    segs, prev, x = [], None, x0
    if show:
        print(f"{'x':>9}{'f(x)':>15}{'знак':>7}")
    while x <= x1 + 1e-12:
        try:
            v = g(x)
            if show:
                print(f"{x:>9.2f}{v:>15.6f}{'+' if v > 0 else '−':>7}")
            if prev is not None and prev[1] * v < 0:
                segs.append((prev[0], x))
            prev = (x, v)
        except (ValueError, ZeroDivisionError):
            if show:
                print(f"{x:>9.2f}{'не определена':>15}")
            prev = None
        x += step
    return segs


print("Вариант 5:  ln(x + 5) = cos x")
print("f(x) = ln(x+5) − cos x,   f′(x) = 1/(x+5) + sin x,   f″(x) = −1/(x+5)² + cos x")
print("Область определения: x > −5.  По условию задачи x < 5.")


Вариант 5:  ln(x + 5) = cos x
f(x) = ln(x+5) − cos x,   f′(x) = 1/(x+5) + sin x,   f″(x) = −1/(x+5)² + cos x
Область определения: x > −5.  По условию задачи x < 5.


In [2]:
# ===== ЗАДАНИЕ 1 — ОТДЕЛЕНИЕ КОРНЕЙ =====
print("=" * 78)
print("ЗАДАНИЕ 1   Отделить корни: графически, программой и в пакете")
print("=" * 78)

print("""
а) ГРАФИЧЕСКИ (схематически, на бумаге)

Разносим уравнение на две кривые:  y₁ = ln(x+5)  и  y₂ = cos x.
Логарифм выходит из −∞ при x → −5 и монотонно растёт.
Косинус всё время колеблется в полосе от −1 до 1.
Значит пересечения возможны, только пока ln(x+5) ≤ 1, то есть
   x + 5 ≤ e,   x ≤ e − 5 ≈ −2,28.
Правее этой точки логарифм уже выше единицы, и корней быть не может.
На участке (−5; −2,28] логарифм пробегает всю полосу [−∞; 1] один раз,
поэтому пересечение ровно одно.
""")

print("б) ПРОГРАММНО — построение и табулирование\n")
plot(f, -4.95, 4.5, title="f(x) = ln(x+5) − cos x   на всей области (−5; 5)")

print("\nТабулирование с шагом 0,25 на участке, где корни возможны:")
segs = tabulate(f, -4.75, -2.25, 0.25)
print(f"\nотрезков со сменой знака: {len(segs)}")
for a, b in segs:
    print(f"   [{a:.2f}; {b:.2f}]:  f({a:.2f}) = {f(a):+.5f},  f({b:.2f}) = {f(b):+.5f}")

A, B = -4.5, -4.2
print(f"\nСужаем до [{A}; {B}] — на нём проверим постоянство знаков производных:")
print(f"{'x':>8}{'f(x)':>13}{'f′(x)':>13}{'f″(x)':>13}")
for i in range(4):
    x = A + (B - A) * i / 3
    print(f"{x:>8.2f}{f(x):>13.6f}{d1(x):>13.6f}{d2(x):>13.6f}")
print("\n   f′ > 0 и f″ < 0 на всём отрезке — корень отделён, метод применим.")

print("\nв) ИНСТРУМЕНТАЛЬНЫМ СРЕДСТВОМ (numpy — сплошное сканирование)\n")
import numpy as np
xs = np.linspace(-4.999, 4.999, 200001)
ys = np.log(xs + 5) - np.cos(xs)
sign_change = np.where(np.sign(ys[:-1]) * np.sign(ys[1:]) < 0)[0]
print(f"   просканировано {len(xs)} точек на (−5; 5)")
print(f"   найдено перемен знака: {len(sign_change)}")
for i in sign_change:
    print(f"      между x = {xs[i]:.5f} и x = {xs[i+1]:.5f}")

print(f"""
ВЫВОД ЗАДАНИЯ 1
Уравнение имеет ровно ОДИН действительный корень, он отделён
на отрезке [{A}; {B}]. Все три способа дали одинаковый ответ.
""")
RESULTS["отрезок"] = (A, B)


ЗАДАНИЕ 1   Отделить корни: графически, программой и в пакете

а) ГРАФИЧЕСКИ (схематически, на бумаге)

Разносим уравнение на две кривые:  y₁ = ln(x+5)  и  y₂ = cos x.
Логарифм выходит из −∞ при x → −5 и монотонно растёт.
Косинус всё время колеблется в полосе от −1 до 1.
Значит пересечения возможны, только пока ln(x+5) ≤ 1, то есть
   x + 5 ≤ e,   x ≤ e − 5 ≈ −2,28.
Правее этой точки логарифм уже выше единицы, и корней быть не может.
На участке (−5; −2,28] логарифм пробегает всю полосу [−∞; 1] один раз,
поэтому пересечение ровно одно.

б) ПРОГРАММНО — построение и табулирование

f(x) = ln(x+5) − cos x   на всей области (−5; 5)
   +3.421 |                                                                    
   +3.034 |                                                      **********    
   +2.646 |                                                  ****          ****
   +2.259 |                                                **                  
   +1.872 |              *****               

In [3]:
# ===== ЗАДАНИЕ 2 — ПОЛОВИННОЕ ДЕЛЕНИЕ, точность 1e-3 =====
print("=" * 78)
print("ЗАДАНИЕ 2   Половинное деление, точность 1e-3")
print("=" * 78)

EPS2 = 1e-3
A, B = RESULTS["отрезок"]
print(f"\nОтрезок [{A}; {B}],  f({A}) = {f(A):+.6f},  f({B}) = {f(B):+.6f}")
print(f"Теоретическое число шагов: log₂(({B}−({A}))/{EPS2:g}) = "
      f"{math.ceil(math.log2((B - A) / EPS2))}")

print("\n" + "-" * 78)
print("а) РУЧНАЯ РАСЧЁТНАЯ ТАБЛИЦА (то, что считается на калькуляторе)")
print("-" * 78)
print(f"\n{'n':>3}{'a':>12}{'b':>12}{'c=(a+b)/2':>13}{'f(c)':>13}{'знак f(c)':>11}{'b−a':>11}")
a, b, fa, n = A, B, f(A), 0
while b - a > EPS2:
    c = (a + b) / 2
    fc = f(c)
    print(f"{n:>3}{a:>12.6f}{b:>12.6f}{c:>13.6f}{fc:>13.6f}"
          f"{('+' if fc > 0 else '−'):>11}{b - a:>11.2e}")
    if fa * fc <= 0:
        b = c
    else:
        a, fa = c, fc
    n += 1
x2 = (a + b) / 2
print(f"\n   ширина отрезка стала меньше {EPS2:g} за {n} шагов")
print(f"   x = {x2:.6f}  →  с требуемой точностью  x ≈ {round(x2, 3)}")

print("\n" + "-" * 78)
print("б) ПРОГРАММА")
print("-" * 78)


def bisect(g, a, b, eps):
    ga, k = g(a), 0
    while b - a > eps:
        c = (a + b) / 2
        gc = g(c)
        if ga * gc <= 0:
            b = c
        else:
            a, ga = c, gc
        k += 1
    return (a + b) / 2, k


xp, np_ = bisect(f, A, B, EPS2)
print(f"\n   bisect(f, {A}, {B}, {EPS2:g})  →  x = {xp:.9f}, шагов {np_}")
print(f"   совпадает с ручным расчётом: {'да' if abs(xp - x2) < 1e-12 else 'нет'}")
print(f"   невязка f(x) = {f(xp):+.3e}")

RESULTS["половинное деление"] = (xp, np_, EPS2)
print(f"\n   ОТВЕТ ЗАДАНИЯ 2:  x ≈ {round(xp, 3)}")


ЗАДАНИЕ 2   Половинное деление, точность 1e-3

Отрезок [-4.5; -4.2],  f(-4.5) = -0.482351,  f(-4.2) = +0.267117
Теоретическое число шагов: log₂((-4.2−(-4.5))/0.001) = 9

------------------------------------------------------------------------------
а) РУЧНАЯ РАСЧЁТНАЯ ТАБЛИЦА (то, что считается на калькуляторе)
------------------------------------------------------------------------------

  n           a           b    c=(a+b)/2         f(c)  знак f(c)        b−a
  0   -4.500000   -4.200000    -4.350000    -0.076274          −   3.00e-01
  1   -4.350000   -4.200000    -4.275000     0.101992          +   1.50e-01
  2   -4.350000   -4.275000    -4.312500     0.014623          +   7.50e-02
  3   -4.350000   -4.312500    -4.331250    -0.030367          −   3.75e-02
  4   -4.331250   -4.312500    -4.321875    -0.007760          −   1.87e-02
  5   -4.321875   -4.312500    -4.317188     0.003459          +   9.38e-03
  6   -4.321875   -4.317188    -4.319531    -0.002143          −   4.69e-03

In [4]:
# ===== ЗАДАНИЕ 3 — ПРОСТАЯ ИТЕРАЦИЯ, точность 1e-6 =====
print("=" * 78)
print("ЗАДАНИЕ 3   Метод простой итерации, точность 1e-6")
print("=" * 78)

EPS3 = 1e-6
A, B = RESULTS["отрезок"]

print("""
ПРИВЕДЕНИЕ К ВИДУ x = φ(x)
Из ln(x+5) = cos x получаем x + 5 = e^(cos x), то есть

        φ(x) = e^(cos x) − 5,      φ′(x) = −sin x · e^(cos x)
""")
phi = lambda x: math.exp(math.cos(x)) - 5
dphi = lambda x: -math.sin(x) * math.exp(math.cos(x))

print(f"Проверка условия сходимости на [{A}; {B}]:")
print(f"{'x':>8}{'φ(x)':>14}{'φ′(x)':>14}")
ds = []
for i in range(7):
    x = A + (B - A) * i / 6
    ds.append(dphi(x))
    print(f"{x:>8.3f}{phi(x):>14.6f}{dphi(x):>14.6f}")
q = max(abs(v) for v in ds)
print(f"\n   φ′ лежит в пределах ({min(ds):.4f}; {max(ds):.4f})")
print(f"   q = max|φ′| = {q:.4f} < 1  — сходимость есть")
print("   φ′ ОТРИЦАТЕЛЬНА: приближения ложатся по разные стороны от корня,")
print("   поэтому критерий |Δ| < ε даёт надёжную оценку без поправки.")

print(f"\nРасчётная таблица, x₀ = {(A + B) / 2:.2f}:")
print(f"{'n':>3}{'xₙ':>16}{'φ(xₙ)':>16}{'|Δ|':>12}{'знак Δ':>9}")
x, n = (A + B) / 2, 0
rows = []
while True:
    y = phi(x)
    d = y - x
    rows.append((n, x, y, abs(d), "+" if d > 0 else "−"))
    if abs(d) < EPS3:
        break
    x = y
    n += 1
for r in rows[:6]:
    print(f"{r[0]:>3}{r[1]:>16.9f}{r[2]:>16.9f}{r[3]:>12.2e}{r[4]:>9}")
print(f"{'...':>3}")
for r in rows[-4:]:
    print(f"{r[0]:>3}{r[1]:>16.9f}{r[2]:>16.9f}{r[3]:>12.2e}{r[4]:>9}")

x3 = rows[-1][2]
print(f"\n   критерий выполнен на шаге {rows[-1][0]},  всего итераций {len(rows)}")
print(f"   x = {x3:.9f},  невязка f(x) = {f(x3):+.3e}")
print("   столбец знаков чередуется — та самая «спираль» из подраздела 2.3")

RESULTS["простая итерация"] = (x3, len(rows), EPS3)
print(f"\n   ОТВЕТ ЗАДАНИЯ 3:  x ≈ {round(x3, 6)}")


ЗАДАНИЕ 3   Метод простой итерации, точность 1e-6

ПРИВЕДЕНИЕ К ВИДУ x = φ(x)
Из ln(x+5) = cos x получаем x + 5 = e^(cos x), то есть

        φ(x) = e^(cos x) − 5,      φ′(x) = −sin x · e^(cos x)

Проверка условия сходимости на [-4.5; -4.2]:
       x          φ(x)         φ′(x)
  -4.500     -4.190061     -0.791740
  -4.450     -4.228477     -0.745116
  -4.400     -4.264594     -0.699814
  -4.350     -4.298482     -0.655956
  -4.300     -4.330215     -0.613634
  -4.250     -4.359872     -0.572908
  -4.200     -4.387533     -0.533811

   φ′ лежит в пределах (-0.7917; -0.5338)
   q = max|φ′| = 0.7917 < 1  — сходимость есть
   φ′ ОТРИЦАТЕЛЬНА: приближения ложатся по разные стороны от корня,
   поэтому критерий |Δ| < ε даёт надёжную оценку без поправки.

Расчётная таблица, x₀ = -4.35:
  n              xₙ           φ(xₙ)         |Δ|   знак Δ
  0    -4.350000000    -4.298482242    5.15e-02        +
  1    -4.298482242    -4.331145832    3.27e-02        −
  2    -4.331145832    -4.310697216   

In [5]:
# ===== ЗАДАНИЕ 4 — КОМБИНИРОВАННЫЙ МЕТОД, точность 1e-6 =====
print("=" * 78)
print("ЗАДАНИЕ 4   Комбинированный метод хорд и касательных, точность 1e-6")
print("=" * 78)

EPS4 = 1e-6
A, B = RESULTS["отрезок"]

print(f"\nНа отрезке [{A}; {B}]:  f′ > 0,  f″ < 0")
print(f"   f({A})·f″({A}) = {f(A) * d2(A):+.4f}  → знаки совпадают → КАСАТЕЛЬНЫЕ отсюда")
print(f"   f({B})·f″({B}) = {f(B) * d2(B):+.4f}  → знаки разные    → ХОРДЫ отсюда")

print(f"\n{'n':>3}{'aₙ (касательная)':>20}{'bₙ (хорда)':>18}{'bₙ − aₙ':>13}{'f(aₙ)':>13}")
a, b, n = A, B, 0
while True:
    print(f"{n:>3}{a:>20.9f}{b:>18.9f}{abs(b - a):>13.2e}{f(a):>13.2e}")
    if abs(b - a) < EPS4:
        break
    a, b = a - f(a) / d1(a), b - f(b) * (b - a) / (f(b) - f(a))
    n += 1

x4 = (a + b) / 2
print(f"\n   корень зажат в [{min(a, b):.9f}; {max(a, b):.9f}]")
print(f"   потребовалось шагов: {n}")
print(f"   x = {x4:.9f},  невязка f(x) = {f(x4):+.3e}")

RESULTS["комбинированный"] = (x4, n, EPS4)
print(f"\n   ОТВЕТ ЗАДАНИЯ 4:  x ≈ {round(x4, 6)}")


ЗАДАНИЕ 4   Комбинированный метод хорд и касательных, точность 1e-6

На отрезке [-4.5; -4.2]:  f′ > 0,  f″ < 0
   f(-4.5)·f″(-4.5) = +2.0311  → знаки совпадают → КАСАТЕЛЬНЫЕ отсюда
   f(-4.2)·f″(-4.2) = -0.5483  → знаки разные    → ХОРДЫ отсюда

  n    aₙ (касательная)        bₙ (хорда)      bₙ − aₙ        f(aₙ)
  0        -4.500000000      -4.200000000     3.00e-01    -4.82e-01
  1        -4.338002854      -4.306922659     3.11e-02    -4.68e-02
  2        -4.318835794      -4.318514729     3.21e-04    -4.79e-04
  3        -4.318635305      -4.318635271     3.42e-08    -5.10e-08

   корень зажат в [-4.318635305; -4.318635271]
   потребовалось шагов: 3
   x = -4.318635288,  невязка f(x) = -1.017e-08

   ОТВЕТ ЗАДАНИЯ 4:  x ≈ -4.318635


In [6]:
# ===== ЗАДАНИЕ 5 — ИНСТРУМЕНТАЛЬНЫЙ ПАКЕТ И СОПОСТАВЛЕНИЕ =====
print("=" * 78)
print("ЗАДАНИЕ 5   Решение в инструментальном пакете и сравнение результатов")
print("=" * 78)

EPS5 = 1e-6
A, B = RESULTS["отрезок"]

print("\nВ задании назван инструментальный пакет; здесь эту роль играет SciPy —")
print("библиотека численных методов с готовыми решателями уравнений.\n")

from scipy.optimize import brentq, newton

x_brent = brentq(f, A, B, xtol=1e-15)
x_newton = newton(f, (A + B) / 2, fprime=d1, tol=1e-15)

print(f"   brentq (гибрид бисекции и обратной параболы): x = {x_brent:.12f}")
print(f"   newton (метод касательных):                   x = {x_newton:.12f}")
print(f"   расхождение между ними: {abs(x_brent - x_newton):.2e}")

X = x_brent          # принимаем за эталон
print(f"\n   эталонное значение корня: x* = {X:.12f}")
print(f"   f(x*) = {f(X):+.2e}")

print("\n" + "=" * 78)
print("СОПОСТАВЛЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 78)
print(f"\n{'метод':<28}{'ε':>10}{'x':>18}{'шагов':>8}{'|x − x*|':>13}")
print("-" * 78)
for name in ("половинное деление", "простая итерация", "комбинированный"):
    v, k, e = RESULTS[name]
    print(f"{name:<28}{e:>10.0e}{v:>18.9f}{k:>8}{abs(v - X):>13.2e}")
print(f"{'SciPy brentq':<28}{'1e-15':>10}{x_brent:>18.9f}{'—':>8}{0.0:>13.2e}")

print(f"""
{'-' * 78}
КОММЕНТАРИЙ

1. Все четыре метода сошлись к одному значению x* ≈ {X:.6f}.
   Различия лежат в пределах заявленной точности каждого — расхождений,
   которые указывали бы на ошибку, нет.

2. Половинное деление дало только три верных знака, но это и требовалось:
   у него была точность 1e-3 против 1e-6 у остальных. Метод надёжен и не
   требует производных, зато на каждый десятичный знак ему нужно около
   3,3 шага, и ускорить его нельзя ничем.

3. Простая итерация при ε = 1e-6 потратила больше всех шагов. Причина в
   том, что q = max|φ′| ≈ 0,79 близко к единице: за шаг ошибка убывает лишь
   на пятую часть. Зато сам метод устроен проще некуда — одна формула,
   производные не нужны.

4. Комбинированный метод выиграл с огромным отрывом: сходимость
   квадратичная, а корень на каждом шаге зажат между касательной и хордой,
   так что ширина этого отрезка сама служит оценкой погрешности. Расплата —
   нужны обе производные и проверка постоянства их знаков на отрезке.

5. Библиотечный brentq по существу делает то же, что и наши методы, но
   переключается между ними автоматически: пока приближение плохое, работает
   бисекция, ближе к корню — быстрая интерполяция. Это объясняет, почему в
   практических расчётах пользуются готовыми пакетами, а разбор методов
   вручную нужен, чтобы понимать, когда такой решатель может подвести.

   ИТОГОВЫЙ ОТВЕТ:  x ≈ {round(X, 6)}
""")


ЗАДАНИЕ 5   Решение в инструментальном пакете и сравнение результатов

В задании назван инструментальный пакет; здесь эту роль играет SciPy —
библиотека численных методов с готовыми решателями уравнений.

   brentq (гибрид бисекции и обратной параболы): x = -4.318635283821
   newton (метод касательных):                   x = -4.318635283821
   расхождение между ними: 0.00e+00

   эталонное значение корня: x* = -4.318635283821
   f(x*) = -9.99e-16

СОПОСТАВЛЕНИЕ РЕЗУЛЬТАТОВ

метод                                ε                 x   шагов     |x − x*|
------------------------------------------------------------------------------
половинное деление               1e-03      -4.318652344       9     1.71e-05
простая итерация                 1e-06      -4.318634987      25     2.96e-07
комбинированный                  1e-06      -4.318635288       3     4.25e-09
SciPy brentq                     1e-15      -4.318635284       —     0.00e+00

---------------------------------------------------